In [ ]:
import numpy as np
import pandas as pd

class Backtester:
    def __init__(self, strategy, historical_data, initial_balance=10000):
        """
        Backtest a strategy with historical data.
        
        strategy: The strategy class instance that implements a `generate_signal` method.
        historical_data: DataFrame containing historical market data with at least 'timestamp', 'close' prices.
        initial_balance: Starting balance for the backtest. Default is 10000.
        """
        self.strategy = strategy
        self.historical_data = historical_data
        self.initial_balance = initial_balance
        self.performance = []
        self.strategy_returns = []
        self.benchmark_returns = []  # Assume you have benchmark data for calculating alpha
        
    def run_backtest(self):
        """
        Run the backtest over the historical data.
        """
        balance = self.initial_balance
        positions = {ticker: None for ticker in self.strategy.config['tickers']}
        for idx, row in self.historical_data.iterrows():
            df = self.historical_data.loc[:idx]
            for ticker in self.strategy.config['tickers']:
                # Generate signal from strategy
                rsi_value = self.strategy.rsi_values[ticker][-1] if ticker in self.strategy.rsi_values else 50
                price = row[ticker]['close']
                # Logic for opening and closing positions based on RSI (or other logic from strategy)
                if rsi_value <= self.strategy.oversold_th['entry'] and positions[ticker] is None:
                    qty = int(balance * self.strategy.free_cash_perc / price)
                    balance -= qty * price
                    positions[ticker] = 'long'
                elif rsi_value >= self.strategy.overbought_th['entry'] and positions[ticker] is None:
                    qty = int(balance * self.strategy.free_cash_perc / price)
                    balance -= qty * price
                    positions[ticker] = 'short'
                elif positions[ticker] == 'long' and rsi_value >= self.strategy.oversold_th['exit']:
                    balance += qty * price
                    positions[ticker] = None
                elif positions[ticker] == 'short' and rsi_value <= self.strategy.overbought_th['exit']:
                    balance += qty * price
                    positions[ticker] = None

            # Track performance and calculate returns for the day
            total_value = balance + sum([qty * row[ticker]['close'] for ticker, qty in positions.items()])
            daily_return = (total_value - self.performance[-1][0]) / self.performance[-1][0] if self.performance else 0
            self.strategy_returns.append(daily_return)
            self.performance.append([total_value, row['timestamp']])

            # Assuming benchmark returns are available for alpha calculation
            benchmark_return = row.get('benchmark', 0)  # Replace 'benchmark' with your actual benchmark data
            self.benchmark_returns.append(benchmark_return)

        # Calculate Sharpe ratio
        sharpe_ratio = np.mean(self.strategy_returns) / np.std(self.strategy_returns) if np.std(self.strategy_returns) != 0 else 0

        # Calculate Alpha
        if len(self.benchmark_returns) > 0:
            covariance = np.cov(self.strategy_returns, self.benchmark_returns)[0][1]
            benchmark_variance = np.var(self.benchmark_returns)
            beta = covariance / benchmark_variance if benchmark_variance != 0 else 0
            alpha = np.mean(self.strategy_returns) - beta * np.mean(self.benchmark_returns)
        else:
            alpha = 0  # No benchmark data

        return {
            'Final Balance': balance,
            'Total Return': (balance - self.initial_balance) / self.initial_balance,
            'Sharpe Ratio': sharpe_ratio,
            'Alpha': alpha
        }